In [ ]:
#Install Solvers
%%capture
import sys
import os

if 'google.colab' in sys.modules:
    !pip install idaes-pse --pre
    !idaes get-extensions --to ./bin
    os.environ['PATH'] += ':bin'

**MODEL 1 (3 AED PLACEMENTS)**

In [ ]:
import pyomo.environ as pyo

rooms_data = [
    ("SCEN0101", 1, 115), ("SCEN0107", 1, 24), ("SCEN0108", 1, 30), ("SCEN0109", 1, 28),
    ("SCEN0110", 1, 40), ("SCEN0111", 1, 30), ("SCEN0112", 1, 25), ("SCEN0113", 1, 25),
    ("SCEN0201", 2, 26), ("SCEN0202", 2, 24), ("SCEN0203", 2, 27), ("SCEN0204", 2, 26),
    ("SCEN0205", 2, 28), ("SCEN0206", 2, 52), ("SCEN0209", 2, 53), ("SCEN0211", 2, 56),
    ("SCEN0212", 2, 40), ("SCEN0320", 3, 28), ("SCEN0322", 3, 27),
    ("SCEN0402", 4, 57), ("SCEN0403", 4, 81), ("SCEN0404", 4, 28), ("SCEN0405", 4, 40),
    ("SCEN0406", 4, 40), ("SCEN0407", 4, 90), ("SCEN0408", 4, 90),
    ("SCEN0501", 5, 57), ("SCEN0503", 5, 23), ("SCEN0504", 5, 23),
    ("SCEN0604", 6, 57), ("SCEN0610", 6, 43), ("SCEN0612", 6, 35), ("SCEN0613", 6, 36),
]

hallway_time = 70
stair_time = 20
time_limit = 180
k = 3

rooms = [r for r, _, _ in rooms_data]
room_floor = {r: f for r, f, _ in rooms_data}
room_cap = {r: c for r, _, c in rooms_data}

# sort rooms on each floor from lowest room number to highest
floor_order = {}
for r, f, _ in rooms_data:
    floor_order.setdefault(f, []).append(r)

for f in floor_order:
    floor_order[f] = sorted(floor_order[f])

room_pos = {r: i for f, rs in floor_order.items() for i, r in enumerate(rs)}
floor_size = {f: len(rs) for f, rs in floor_order.items()}

travel = {}
for i in rooms:
    for j in rooms:
        fi, fj = room_floor[i], room_floor[j]

        if fi == fj:
            n = floor_size[fi]
            if n == 1:
                travel[i, j] = 0
            else:
                step_time = hallway_time / (n - 1)
                travel[i, j] = step_time * abs(room_pos[i] - room_pos[j])
        else:
            travel[i, j] = hallway_time + stair_time * abs(fi - fj)

can_cover = {(i, j): int(travel[i, j] <= time_limit) for i in rooms for j in rooms}

model = pyo.ConcreteModel()
model.I = pyo.Set(initialize=rooms)
model.J = pyo.Set(initialize=rooms)

model.x = pyo.Var(model.J, domain=pyo.Binary)
model.y = pyo.Var(model.I, model.J, domain=pyo.Binary)

model.obj = pyo.Objective(
    expr=sum(room_cap[i] * travel[i, j] * model.y[i, j] for i in model.I for j in model.J),
    sense=pyo.minimize
)

model.num_aeds = pyo.Constraint(expr=sum(model.x[j] for j in model.J) == k)
model.assign = pyo.Constraint(model.I, rule=lambda m, i: sum(m.y[i, j] for j in m.J) == 1)
model.open_link = pyo.Constraint(model.I, model.J, rule=lambda m, i, j: m.y[i, j] <= m.x[j])
model.time_limit = pyo.Constraint(model.I, model.J, rule=lambda m, i, j: m.y[i, j] <= can_cover[i, j])

solver = pyo.SolverFactory("highs")
results = solver.solve(model)

chosen = [j for j in model.J if pyo.value(model.x[j]) > 0.5]
print("Chosen AED locations:", chosen)

for i in model.I:
    for j in model.J:
        if pyo.value(model.y[i, j]) > 0.5:
            print(f"{i} -> {j} ({travel[i, j]:.2f} sec)")
            break

obj_val = pyo.value(model.obj)
print("Weighted total response time =", obj_val)
print("Average response time =", obj_val / sum(room_cap.values()))

Chosen AED locations: ['SCEN0108', 'SCEN0206', 'SCEN0406']
SCEN0101 -> SCEN0108 (20.00 sec)
SCEN0107 -> SCEN0108 (10.00 sec)
SCEN0108 -> SCEN0108 (0.00 sec)
SCEN0109 -> SCEN0108 (10.00 sec)
SCEN0110 -> SCEN0108 (20.00 sec)
SCEN0111 -> SCEN0108 (30.00 sec)
SCEN0112 -> SCEN0108 (40.00 sec)
SCEN0113 -> SCEN0108 (50.00 sec)
SCEN0201 -> SCEN0206 (43.75 sec)
SCEN0202 -> SCEN0206 (35.00 sec)
SCEN0203 -> SCEN0206 (26.25 sec)
SCEN0204 -> SCEN0206 (17.50 sec)
SCEN0205 -> SCEN0206 (8.75 sec)
SCEN0206 -> SCEN0206 (0.00 sec)
SCEN0209 -> SCEN0206 (8.75 sec)
SCEN0211 -> SCEN0206 (17.50 sec)
SCEN0212 -> SCEN0206 (26.25 sec)
SCEN0320 -> SCEN0406 (90.00 sec)
SCEN0322 -> SCEN0206 (90.00 sec)
SCEN0402 -> SCEN0406 (46.67 sec)
SCEN0403 -> SCEN0406 (35.00 sec)
SCEN0404 -> SCEN0406 (23.33 sec)
SCEN0405 -> SCEN0406 (11.67 sec)
SCEN0406 -> SCEN0406 (0.00 sec)
SCEN0407 -> SCEN0406 (11.67 sec)
SCEN0408 -> SCEN0406 (23.33 sec)
SCEN0501 -> SCEN0406 (90.00 sec)
SCEN0503 -> SCEN0406 (90.00 sec)
SCEN0504 -> SCEN0406 (